# Deepfake Hybrid Detection — Colab Run
**Before running:** `Runtime → Change runtime type → T4 GPU`

Edit the two paths in **Step 1**, then run cells top to bottom.

- **Normal run:** Steps 1 → 2 → 3 → 4 (run_pipeline) → 5 (save)
- **If Step 4 fails mid-way:** use Step 4b (individual steps) to re-run only the failing phase

## Step 1 — Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these values before running
# ═══════════════════════════════════════════════════════════════════════════

# Paths
FFPP_IN_DRIVE = '/content/drive/MyDrive/ffpp'   # folder with original_sequences/ and manipulated_sequences/
PROJECT       = '/content/skripsi/deepfake_hybrid'
CONFIG        = f'{PROJECT}/config.yaml'
FFPP_LOCAL    = '/content/ffpp'
CDF_LOCAL     = '/content/celeb_df'

# Dataset download
FFPP_NUM_REAL = None   # number of real videos to download (None = all ~1000)
FFPP_NUM_FAKE = None   # number of fake videos per method to download (None = all ~1000 each)
FFPP_SERVER   = 'EU2'  # EU, EU2, or CA — switch if download is slow

# Training — use a list to train multiple sample sizes in sequence
N_SAMPLES_LIST = [50, 200, 400]  # matches BAB III: 3 variasi ukuran dataset
MAX_FRAMES    = 100    # max frames extracted per video
EPOCHS        = 10
BATCH_SIZE    = 16     # reduce to 8 if CUDA OOM
NUM_WORKERS   = 2
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
IMAGE_SIZE    = 224
N_SEEDS       = 1      # increase to 3 for statistical validity
PRETRAINED    = True   # use ImageNet-pretrained XceptionNet

# Preprocessing
RECOMPUTE_FFT = True   # set to True to force recompute FFT cache (needed after changing fft_utils.py)

## Step 2 — Setup

_(command.txt §1)_

In [ ]:
import os
os.chdir('/')  # move to safe dir before deleting /content/skripsi

!rm -rf /content/skripsi
!git clone https://github.com/giovannyhalimko/skripsi.git /content/skripsi

!pip install --upgrade pip -q
!pip install -r {PROJECT}/requirements.txt -q
!apt-get install -qq ffmpeg

import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 3 — Copy datasets & patch config

In [ ]:
import yaml, os

# Patch config first so the download script writes to the right place
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

cfg['datasets']['ffpp']['root'] = FFPP_LOCAL
cfg['datasets']['cdf']['root']  = CDF_LOCAL
cfg['datasets']['cdf']['real_keywords'] = ['real', 'authentic', 'youtube']
cfg['output_root'] = '/content/outputs'
cfg['epochs']      = EPOCHS
cfg['num_workers'] = NUM_WORKERS
cfg['batch_size']  = BATCH_SIZE

with open(CONFIG, 'w') as f:
    yaml.dump(cfg, f)
print('Config patched. Downloading FFPP...')

# Download FFPP directly in Colab
real_flag = f'--num-videos {FFPP_NUM_REAL}' if FFPP_NUM_REAL else ''
fake_flag = f'--num-videos {FFPP_NUM_FAKE}' if FFPP_NUM_FAKE else ''

!echo "" | python {PROJECT}/scripts/download_datasets.py --config {CONFIG} \
    --datasets original --compression c23 --type videos \
    {real_flag} --server {FFPP_SERVER}

!echo "" | python {PROJECT}/scripts/download_datasets.py --config {CONFIG} \
    --datasets Deepfakes Face2Face FaceSwap NeuralTextures \
    --compression c23 --type videos \
    {fake_flag} --server {FFPP_SERVER}

!echo "FFPP real:" $(find {FFPP_LOCAL}/original_sequences -name "*.mp4" | wc -l) videos
!echo "FFPP fake:" $(find {FFPP_LOCAL}/manipulated_sequences -name "*.mp4" | wc -l) videos

os.chdir(PROJECT)

In [ ]:
# Back up downloaded FFPP to Drive for future runs (skips files already there)
!cp -rn {FFPP_LOCAL}/ "/content/drive/MyDrive/ffpp/"
!echo "FFPP saved to Drive."

In [ ]:
# Download Celeb-DF v2 from Google Drive (with progress bar)
!pip install -q gdown
import os
os.makedirs(f"{PROJECT}/dataset/celeb_df", exist_ok=True)
!gdown "1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj" -O "{PROJECT}/dataset/celeb_df/Celeb-DF-v2.zip"
print("Celeb-DF downloaded.")

In [ ]:
# CDF — unzip from project folder
import shutil
from pathlib import Path

!mkdir -p {CDF_LOCAL}
!unzip -q "{PROJECT}/dataset/celeb_df/Celeb-DF-v2.zip" -d {CDF_LOCAL}

root = Path(CDF_LOCAL)
expected = {'Celeb-real', 'YouTube-real', 'Celeb-synthesis'}
subdirs = [d for d in root.iterdir() if d.is_dir()]
if subdirs and not expected.intersection({d.name for d in subdirs}):
    for item in subdirs[0].iterdir():
        shutil.move(str(item), str(root / item.name))
    subdirs[0].rmdir()

!ls {CDF_LOCAL}

## Step 4 — Run everything

_(command.txt §2 — extract frames → splits → FFT cache → train → eval → tables)_

In [ ]:
import yaml, os
from datetime import datetime

# Write a fresh colab config with correct absolute paths
colab_cfg = {
    'output_root': '/content/outputs',
    'frame_sampling_fps': 5,
    'max_frames_per_video': MAX_FRAMES,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'epochs': EPOCHS,
    'fusion_mode': 'two_branch',
    'n_seeds': N_SEEDS,
    'datasets': {
        'ffpp': {
            'root': FFPP_LOCAL,
            'real_keywords': ['original', 'real', 'pristine', 'actors', 'youtube'],
            'fake_keywords': ['fake', 'manipulated', 'deepfakes', 'faceswap',
                              'neuraltextures', 'deepfakedetection', 'faceshifter', 'face2face'],
        },
        'cdf': {
            'root': CDF_LOCAL,
            'real_keywords': ['real', 'authentic', 'youtube'],
            'fake_keywords': ['fake', 'synthesis', 'deepfake'],
        }
    }
}

COLAB_CONFIG = f'{PROJECT}/colab_config.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(colab_cfg, f, default_flow_style=False)

# Verify paths before running
with open(COLAB_CONFIG) as f:
    check = yaml.safe_load(f)
print("FFPP root:", check['datasets']['ffpp']['root'])
print("CDF  root:", check['datasets']['cdf']['root'])
assert check['datasets']['ffpp']['root'] == FFPP_LOCAL, "FFPP path wrong!"

os.chdir(PROJECT)
pretrained_flag = '--pretrained' if PRETRAINED else ''
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_base = f'/content/drive/MyDrive/skripsi_outputs/{ts}'

for i, n_samples in enumerate(N_SAMPLES_LIST):
    print(f"\n{'#' * 60}")
    print(f"  Training session {i+1}/{len(N_SAMPLES_LIST)}: N={n_samples}")
    print(f"{'#' * 60}\n")

    # Only force FFT recompute on first iteration
    force_fft_flag = '--force-fft' if (RECOMPUTE_FFT and i == 0) else ''

    !python scripts/run_pipeline.py --config colab_config.yaml --n-samples {n_samples} --max-frames {MAX_FRAMES} --epochs {EPOCHS} {force_fft_flag} {pretrained_flag}

    # Save intermediate results to Drive
    dst = f'{drive_base}/n{n_samples}'
    !mkdir -p "{dst}"
    !cp -r /content/outputs/tables/n{n_samples} "{dst}/tables" 2>/dev/null; true
    !cp -r /content/outputs/runs/*_n{n_samples}_* "{dst}/runs/" 2>/dev/null; true
    print(f"  Saved N={n_samples} results to Drive: {dst}")

# Generate plots from all results
n_samples_str = ','.join(str(n) for n in N_SAMPLES_LIST)
!python scripts/plot_results.py --config colab_config.yaml --n-samples {n_samples_str}

# Save plots to Drive
!cp -r /content/outputs/plots "{drive_base}/plots" 2>/dev/null; true

print(f"\n{'#' * 60}")
print(f"  ALL SESSIONS COMPLETE — trained N={N_SAMPLES_LIST}")
print(f"  Results saved to: {drive_base}")
print(f"{'#' * 60}")

---
## Step 4b — Individual steps (only if Step 4 fails)

_(command.txt §3)_

Run only the cell for the phase that failed — skip the ones that already completed.

In [ ]:
# Extract frames
!python scripts/extract_frames.py --config {CONFIG} --dataset FFPP --fps 5 --max-frames {MAX_FRAMES} --n-samples {N_SAMPLES}
!python scripts/extract_frames.py --config {CONFIG} --dataset CDF  --fps 5 --max-frames {MAX_FRAMES} --n-samples {N_SAMPLES}

In [ ]:
# Build splits
!python scripts/build_splits.py --config {CONFIG} --dataset FFPP
!python scripts/build_splits.py --config {CONFIG} --dataset CDF

In [ ]:
# FFT cache
force_flag = '--force' if RECOMPUTE_FFT else ''
!python scripts/compute_fft_cache.py --config {CONFIG} --dataset FFPP --num-workers {NUM_WORKERS} {force_flag}
!python scripts/compute_fft_cache.py --config {CONFIG} --dataset CDF  --num-workers {NUM_WORKERS} {force_flag}

In [ ]:
# Train individual models
pretrained_flag = '--pretrained' if PRETRAINED else ''
!python scripts/train.py --config {CONFIG} --dataset FFPP --model spatial {pretrained_flag}
!python scripts/train.py --config {CONFIG} --dataset FFPP --model freq
!python scripts/train.py --config {CONFIG} --dataset FFPP --model hybrid  {pretrained_flag}

In [ ]:
# Full matrix: train + eval + tables
pretrained_flag = '--pretrained' if PRETRAINED else ''
!python scripts/run_all.py --config {CONFIG} --n-samples {N_SAMPLES} {pretrained_flag}

---
## Step 5 — Save outputs to Drive

**Run before closing the session.**

Output locations (command.txt §Output locations):
- `outputs/tables/Table1_in_dataset.csv`, `Table2_cross_dataset.csv`, `Table3_generalization_drop.csv`
- `outputs/runs/<model>_<dataset>_n<N>_seed<N>/best.pt`
- `outputs/runs/<model>_<dataset>_n<N>_seed<N>/train.log`

In [ ]:
from datetime import datetime
ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/content/drive/MyDrive/skripsi_outputs/{ts}'

!cp -r /content/outputs "{dst}"
!echo "Saved to: {dst}"
!ls "{dst}"